In [ ]:
"""Settings related to escape pattern analysis."""
Path = ''


In [ ]:
"""Imports"""
import warnings
import numpy as np
import seaborn as sns
np.warnings = warnings
from replay_trajectory_classification import SortedSpikesDecoder, Environment, RandomWalk, estimate_movement_var

In [ ]:
# load data
data = np.load(Path)
sampling_frequency = 1000 / data['bin_width'] # in Hz, bin_width is in ms

# set up environment parameters
linear_track_env = Environment(
                            place_bin_size=1,  # Adjust based on your actual bin size
                            # track_graph=my_linear_graph,  # Define a linear graph with 25 nodes
                            edge_order=[(i, i + 1) for i in range(24)],  # Sequential edges
                            edge_spacing=None,  # No gaps between edges
                            position_range=[(0, 25)],  # 0 to 100% of escape route
                            infer_track_interior=False,  # Data is already binned
                            fill_holes=False,
                            dilate=False,
                            bin_count_threshold=1,
                        )

transition_type = RandomWalk(movement_var=1)

decoder = SortedSpikesDecoder(
                            environment=linear_track_env,
                            transition_type=transition_type,
                            sorted_spikes_algorithm='spiking_likelihood_kde',
                            sorted_spikes_algorithm_params={'block_size': None,
                                                            'position_std': [1.0],
                                                            'use_diffusion': False},
                        )

decoder.fit(data['train_position'], data['train_spikes'])

results = decoder.predict(data['test_spikes'], time=data['test_time'])

# save results
filename = Path + '_replay_results.npz'
np.savez(filename,
         causal_posterior=results.causal_posterior,
         acausal_posterior=results.acausal_posterior,
         )

In [ ]:
"""Plot train and test data predictions from decoder"""
# use only either causal or acausal posterior for plotting
# also plot the actual run trajectory?

In [ ]:
"""Look at the actual and decoder tuning curves"""
g = (decoder.place_fields_ * sampling_frequency).plot(
        x="position", col="neuron", col_wrap=5, color="red", linewidth=2, alpha=0.9, zorder=1, label="Predicted")
g.axes[0, 0].set_ylabel("Firing Rate [spikes/s]")
for ax, place_field in zip(g.axes.flat, tuning_curves):
    ax.plot(np.arange(25), place_field, linewidth=2, color="black", zorder=-1, label="True")
sns.despine()